# CIR-ARC Phase 2: Object-Centric Neural Perception on Google Colab

This notebook runs the complete **Phase 2 Neural Perception Training Pipeline** with Slot Attention on Google Colab GPU (T4 / V100 / A100).

### Pipeline Components:
1. **ColorEmbedding (48-dim)**: Discrete ARC palette (0-9 + pad token 10) to continuous latent space
2. **CNNStem (128-dim)**: Spatial feature extraction without pooling (preserves exact cell coordinates)
3. **SlotAttention (24 slots, 3 iters)**: Competitive binding to discover discrete objects unsupervised/weakly-supervised
4. **PropertyHeads**: Parallel MLPs predicting color, shape, size, position, orientation, symmetry
5. **ReconstructionDecoder**: Positional attention decoding back to 2D grid color logits

## 1. Hardware & Environment Check

In [ ]:
# Check assigned GPU
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive (For Persistent Checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/CIR_ARC_checkpoints/phase2'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to Google Drive: {CHECKPOINT_DIR}")

## 3. Clone Repository & Install Package

In [ ]:
# Clone repo (Replace with your GitHub repository URL if private)
%cd /content
!rm -rf CIR-ARC
!git clone https://github.com/YOUR_GITHUB_USERNAME/CIR-ARC.git
%cd CIR-ARC

# Install in editable mode
!pip install -e .
!pip install scipy matplotlib pyyaml tqdm

## 4. Run Neural Unit & Invariant Tests (Verify Setup)

In [ ]:
!pytest -q

## 5. Generate Synthetic Training & Held-Out Datasets

In [ ]:
# Generate 1,000 tasks per rule + compositions directly on Colab high-speed local disk
!python scripts/generate_data.py --n_per_rule 1000 --n_per_pair 500 --output_dir data/synthetic

## 6. Train Perception Model (Phase 2 Full Training)

In [ ]:
!python scripts/train_perception.py \
    --config configs/phase2.yaml \
    --epochs 30 \
    --batch_size 32 \
    --lr 0.001 \
    --device cuda \
    --checkpoint_dir /content/drive/MyDrive/CIR_ARC_checkpoints/phase2

## 7. Interactive Visual Inspection: Slot Decomposition & Reconstructions

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import torch
from cir_arc.neural.training.trainer import PerceptionModel
from cir_arc.neural.training.dataset import SyntheticArcDataset, collate_variable_grids

# Standard ARC Color Palette
ARC_COLORS = [
    '#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
    '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'
]
cmap = mcolors.ListedColormap(ARC_COLORS)
norm = mcolors.Normalize(vmin=0, vmax=9)

# Load best trained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PerceptionModel().to(device)

best_ckpt = '/content/drive/MyDrive/CIR_ARC_checkpoints/phase2/phase2_slot_attention_baseline/best_model.pt'
if os.path.exists(best_ckpt):
    state = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(state['model_state_dict'] if 'model_state_dict' in state else state)
    print("Loaded trained weights from Google Drive!")
else:
    print("Using model directly (evaluate current state).")

model.eval()

# Load sample batch from validation set
val_ds = SyntheticArcDataset(data_dir="data/synthetic/held_out")
batch = collate_variable_grids([val_ds[i] for i in range(min(4, len(val_ds)))])

grids = batch['input_grids'].to(device)
masks = batch['input_masks'].to(device)

with torch.no_grad():
    out = model(grids, mask=masks)

recon_preds = out['recon_logits'].argmax(dim=-1).cpu().numpy()
attn_maps = out['attn_maps'].cpu().numpy()  # (B, K, H*W)
objectness = out['objectness'].cpu().numpy()

# Plot Input vs Reconstruction vs Top Slots
sample_idx = 0
H, W = batch['heights'][sample_idx], batch['widths'][sample_idx]
input_grid = grids[sample_idx, :H, :W].cpu().numpy()
recon_grid = recon_preds[sample_idx, :H, :W]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(input_grid, cmap=cmap, norm=norm)
axes[0].set_title("Ground Truth Input")
axes[1].imshow(recon_grid, cmap=cmap, norm=norm)
axes[1].set_title("Slot Reconstruction")

# Top 2 active object slots
top_slots = np.argsort(objectness[sample_idx])[::-1][:2]
for i, slot_k in enumerate(top_slots):
    slot_attn = attn_maps[sample_idx, slot_k].reshape(grids.shape[1], grids.shape[2])[:H, :W]
    axes[2 + i].imshow(slot_attn, cmap='magma')
    axes[2 + i].set_title(f"Slot {slot_k} Attn (Obj={objectness[sample_idx, slot_k]:.2f})")

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()